In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 274 nodes, deleted 355 relationships, completed after 3867 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 5791 ms.


### Node Rules 

In [4]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env.json")
# env = Environment("../dtgraph/type_checking/env_common_movies.json")

# env = Environment(
#     "../dtgraph/type_checking/env.json",
#     functions_path="../dtgraph/type_checking/functions.json"
# )

#################################################
old_node_rule = Rule("""
MATCH (p:Person)
GENERATE
(x = (p):Actor {
    name = p.name,
    born = p.born,
    score = p.born * (p.born + 2),
    experienced = p.born < 1950
})
""", env=env, type_strict=True)

old_edge_rule = Rule(
    """
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name,
    years = [y.released],
    movies = [y.title] 
})
""",
    env=env,
    type_strict=True,
)

Rule_HAS_A = Rule(
"""
MATCH (m:Movie)<-[:ACTED_IN]-(p:Person)
GENERATE
((m):Movie {
    movie = m.title
})-[():HAS_A {
    movies = [p.name]
}]->(():Actor{
name = p.name })
""",
env=env,
type_strict=True,
)


Rule_PAIR = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[(x,y):ACTED_IN_PAIR {
    movies = [y.title],
    years = [y.released]
}]->((y):Movie {
    movie = y.title
})
""",
env=env,
type_strict=True,
)
#################################################

Rule_TEST2 = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
(u = (x):Person {
    name = x.name
})-[(x):IS_AN {
    years = [y.released],
    movies = [y.title]
}]->(z = ():Actor)
""",env=env,type_strict=True,
)

#################################################

Rule_one = Rule(
"""
MATCH (p:Person)
GENERATE
((_):Actor {
    name = p.name
})
""",
env=env,
type_strict=True,
)

Rule_two = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[():WORKED_ON {
    movie = y.title
}]->((y):Movie {
    movie = y.title
})
""",
env=env,
type_strict=True,
)

Rule_Three = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[():IS_AN {
    movies = [y.title]
}]->((_):ActorGroup)
""",
env=env,
type_strict=True,
)


Rule_four = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((_):TempActor {
    name = x.name
})-[():TEMP_REL {
    movie = y.title
}]->((_):TempMovie {
    movie = y.title
})
""",
env=env,
type_strict=True,
)
#############################################


####
test_one_A = Rule(
"""
MATCH (p:Person)
GENERATE
(("const1"):TestA {
    value = "A"
})
""",
env=env,
type_strict=False,
)

test_one_B = Rule(
"""
MATCH (p:Person)
GENERATE
(("const1"):TestB {
    value = "B"
})
""",
env=env,
type_strict=False,
)
####


### Execute Rules

In [5]:
my_transform = Transformation([Rule_TEST2])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 370 ms.
prop-value [y.released]
AST:
ListExpression
    └── PropertyAccess
        ├── var: y
        └── prop: released
prop-value [y.title]
AST:
ListExpression
    └── PropertyAccess
        ├── var: y
        └── prop: title
Rule: Added 206 labels, created 103 nodes, set 721 properties, created 102 relationships, completed after 2959 ms.


2959

### Abort Transformation

In [77]:
my_transform.abort()

Index: Removed 1 index, completed after 57 ms.
Abort: Deleted 0 nodes, deleted 0 relationships, completed after 66 ms.
